# Parallel-beam reconstruction (FBP / SIRT)

Reconstructs a **parallel-beam re-projection** of a trained NAF model, i.e. the `.nii` written by `../CBCT/reproject_parallel.py` or `../msCBCT/reproject_parallel.py`.

A parallel beam has no scanner geometry: at angle `t` the rays are `(-sin t, cos t, 0)`. There is no source, no SOD/SDD magnification, no half-fan offset and no cone angle, so the inputs are only

| Source | Provides |
|---|---|
| the `.nii` itself | number of views, detector rows x columns, pixel width (= default voxel size) |
| `params.py` | **SOD and SDD only**, for the FOV radius `R = SOD * sin(atan(u_far / SDD))`; never a ray direction |
| the parameter cell below | angular span / start / rotation direction, algorithm, output |

The `*_geom.json` sidecar written next to the re-projection is only **cross-checked** and printed, never used as input.

* **FBP is exact** for a parallel beam: the rays are perpendicular to z, so each detector row is an independent 2D parallel sinogram of its own slice (no cone weighting, no redundancy weighting, no Parker window). 1440 views take about 13 s on an RTX 5090.
* **SIRT** adds non-negativity and gives fewer streaks.

Outputs in `OUT_DIR`, with `<stem> = recon_<name>_<algorithm>` and `<name>` the input file name without `.nii` (and without a leading `proj_`):

* `<stem>.nii`: linear attenuation coefficient mu [cm^-1]
* `<stem>_HU.nii`: Hounsfield units, `HU = (mu - mu_water) / (mu_water - mu_air) * 1000`, with mu_water / mu_air at the mean beam energy of the scan the model was trained on (msCBCT 67 keV, CBCT 95 keV)
* `<stem>_HU_dicom/`: the same HU as a CT DICOM series, one file per axial slice

The `.nii` files are float32 with NIfTI dims (x, y, z) and the voxel width in the header.

## 1. Imports

Run the notebook from this folder (`reconstruction/`), so that `astra_recon.py`, `hu.py` and `params.py` are importable.

In [ ]:
import astra
from astra_recon import ParallelReconstruction

print(f"ASTRA {astra.__version__}  |  CUDA: {astra.use_cuda()}")

## 2. Parameters (the only cell you normally edit)

* `SPAN_DEG` / `START_DEG` / `ROT_DIR` must match the re-projection: `reproject_parallel.py` always writes 360 deg starting at 0 deg, with the training config's `rot_dir`.
* `IMG_SIZE = None` gives (FOV diameter, FOV diameter, detector height); `(240.0, 240.0, 120.0)` is the NAF training grid, for voxel-by-voxel comparison with the NAF volume.
* `MEAN_ENERGY_KEV` and `DEVICE` describe the scan the re-projected model was trained on: msCBCT 67 keV, CBCT 95 keV. `PHANTOM`, `KVP` and `TUBE_CURRENT_MA` only fill DICOM tags.

In [ ]:
# ---- input: a parallel-beam re-projection written by reproject_parallel.py ----
PROJ_FILE  = "../msCBCT/output/RANDO_msCBCT_360views_12000_par3d_1440.nii"
SPAN_DEG   = 360.0          # [deg] span of all views (re-projection default: 360)
START_DEG  = 0.0            # [deg] angle of the first view
ROT_DIR    = +1             # rotation sense, as the training config's rot_dir
# ---- reconstruction ----
ALGORITHM  = "FBP"          # "FBP" | "SIRT"
FBP_FILTER = "shepp-logan"  # ram-lak / shepp-logan / cosine / hamming / hann
ITERATIONS = 300            # SIRT only
IMG_SIZE   = (240.0, 240.0, 120.0)  # [mm] (x, y, z); None = (FOV diam., FOV diam., detector height)
VOXEL_SIZE = 0.4            # [mm]; None = detector pitch
FOV_MARGIN = 1.3            # scales the measured FOV radius (>= reproject_parallel.py's fov_margin_xy)
FOV_MASK   = True           # zero outside the FOV cylinder and the detector's z range
# ---- mu -> HU and DICOM tags: the scan the model was trained on ----
MEAN_ENERGY_KEV = 67.0      # [keV] mean beam energy: msCBCT 67, CBCT 95
DEVICE          = "msCBCT"  # DICOM device tag: "msCBCT" or "CBCT"
PHANTOM         = None      # DICOM patient name / ID; None = the input file name
KVP             = 110       # DICOM tag
TUBE_CURRENT_MA = 11        # DICOM tag [mA]
# ---- output ----
OUT_DIR    = "./output"
GPU        = None           # None = ASTRA's default GPU

## 3. Load the projection

Prints the views, detector, FOV and volume size. Check the `views` line against the re-projection, and the cross-check line if a `*_geom.json` sidecar is present.

In [ ]:
rec = ParallelReconstruction.from_file(PROJ_FILE, span_deg=SPAN_DEG, start_deg=START_DEG,
                                       rot_dir=ROT_DIR, img_size=IMG_SIZE,
                                       voxel_size=VOXEL_SIZE, fov_margin=FOV_MARGIN)

## 4. Reconstruct and show the central slices

With `FOV_MASK` the volume is zeroed outside the FOV cylinder and outside the detector's z range, where the reconstruction has no data.

In [ ]:
rec.run(algorithm=ALGORITHM, fbp_filter=FBP_FILTER, iterations=ITERATIONS,
        fov_mask=FOV_MASK, gpu=GPU)
rec.show_slices()

## 5. Save

Writes mu (`.nii`), HU (`_HU.nii`) and the HU DICOM series (`_HU_dicom/`) to `OUT_DIR`.

In [ ]:
rec.save(OUT_DIR, MEAN_ENERGY_KEV, phantom=PHANTOM, kvp=KVP,
         tube_current_ma=TUBE_CURRENT_MA, device=DEVICE)